In [1]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

try:
    import mlflow
    print("MLflow:", mlflow.__version__)
    print("MLflow: OK")
except Exception as e:
    print("MLflow ERROR:", repr(e))

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Executable: C:\Users\tanak\anaconda3\python.exe
MLflow: 3.16.1
MLflow: OK


# Part 3 – Machine Learning and AI

## Task 4 – Advanced AI Technique: MLflow Experiment Tracking

### Objective

MLflow is selected as the advanced AI technique for this task.

The purpose is to improve experiment reproducibility and model management by recording model configurations, parameters, evaluation metrics and model artifacts in a structured experiment-tracking system.

Two traffic-volume regression models are tracked:

1. Linear Regression – baseline model
2. Random Forest Regressor – non-linear ensemble model

Both models use the same engineered traffic dataset and feature set so that their performance can be compared consistently.

MLflow is particularly suitable for this capstone because the experiment records can later support the MLOps requirements in Task 6, including experiment tracking and model versioning.

In [2]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.propagate = False

if logger.handlers:
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(
    "task4_mlflow.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)


logger.info(
    "Task 4 MLflow experiment-tracking workflow started."
)

print("MLflow version:", mlflow.__version__)

2026-10-07 22:14:03,250 - INFO - __main__ - Task 4 MLflow experiment-tracking workflow started.


MLflow version: 3.16.1


## 4.3 Load the Engineered Dataset

The engineered traffic dataset produced in Part 2 is reused.

The regression problem is kept consistent with Task 1: the objective is to predict `traffic_volume` using engineered time, weather and holiday-related features.


In [3]:
DATA_PATH = Path(
    "../part2_python/engineered_traffic_data.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date_time"]
)

logger.info(
    "Dataset loaded successfully: %d rows, %d columns.",
    df.shape[0],
    df.shape[1]
)

print("Dataset shape:", df.shape)

2026-10-07 22:14:34,840 - INFO - __main__ - Dataset loaded successfully: 48187 rows, 30 columns.


Dataset shape: (48187, 30)


## 4.4 Prepare the Regression Experiment

The experiment uses the same traffic-volume prediction problem developed in Task 1.

Time, weather, holiday and cyclical features are used as predictors. The target variable is `traffic_volume`.

A fixed random train/test split is used to make the experiment reproducible. Linear Regression uses standardised predictors, while Random Forest uses the original feature values.

The purpose of this task is not to introduce a new predictive problem, but to demonstrate how MLflow can systematically track and compare machine learning experiments.

In [4]:
# ------------------------------------------------------------
# Additional features
# ------------------------------------------------------------

holiday_text = (
    df["holiday"]
    .fillna("None")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["is_holiday"] = (
    ~holiday_text.isin(
        ["none", "nan", ""]
    )
).astype(int)

df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)


# ------------------------------------------------------------
# Weather features
# ------------------------------------------------------------

weather_features = [
    column
    for column in df.columns
    if column.startswith("weather_")
    and column not in [
        "weather_main",
        "weather_description"
    ]
]


base_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "has_rain",
    "has_snow"
]


feature_columns = []

for column in base_features + weather_features:
    if (
        column in df.columns
        and column not in feature_columns
    ):
        feature_columns.append(column)


X = df[feature_columns].copy()
y = df["traffic_volume"].copy()


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


logger.info(
    "MLflow experiment data prepared with %d predictors.",
    len(feature_columns)
)

print("Number of predictors:", len(feature_columns))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

2026-10-07 22:15:22,028 - INFO - __main__ - MLflow experiment data prepared with 25 predictors.


Number of predictors: 25
Training rows: 38549
Test rows: 9638


## 4.5 Configure MLflow Experiment Tracking

A local MLflow tracking store is configured for the capstone.

The experiment is named `capstone_traffic_volume_prediction`. Each model run records its algorithm, parameters, evaluation metrics and model artifact.

Keeping the tracking data within the project provides reproducible evidence of the experiments and allows the runs to be inspected later using the MLflow user interface.

In [6]:
# ------------------------------------------------------------
# Configure local MLflow tracking
# ------------------------------------------------------------

MLFLOW_DB = Path(
    "mlflow_tracking.db"
).resolve()

TRACKING_URI = (
    f"sqlite:///{MLFLOW_DB.as_posix()}"
)

mlflow.set_tracking_uri(
    TRACKING_URI
)

EXPERIMENT_NAME = (
    "capstone_traffic_volume_prediction"
)

mlflow.set_experiment(
    EXPERIMENT_NAME
)


logger.info(
    "MLflow configured: experiment=%s.",
    EXPERIMENT_NAME
)


print("Experiment:", EXPERIMENT_NAME)
print("Tracking database:", MLFLOW_DB)
print("Tracking URI:", mlflow.get_tracking_uri())

2026/10/07 22:16:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/07 22:16:11 INFO mlflow.store.db.utils: Updating database tables
2026/10/07 22:16:14 INFO mlflow.tracking.fluent: Experiment with name 'capstone_traffic_volume_prediction' does not exist. Creating a new experiment.
2026-10-07 22:16:14,049 - INFO - __main__ - MLflow configured: experiment=capstone_traffic_volume_prediction.


Experiment: capstone_traffic_volume_prediction
Tracking database: C:\Users\tanak\Documents\GitHub\smart-city-traffic-capstone\part3_machine_learning\mlflow_tracking.db
Tracking URI: sqlite:///C:/Users/tanak/Documents/GitHub/smart-city-traffic-capstone/part3_machine_learning/mlflow_tracking.db


## 4.6 Track the Linear Regression Baseline

The Linear Regression baseline is trained and recorded as the first MLflow run.

The run records:

- Model algorithm
- Train/test configuration
- Number of predictors
- MAE
- R-squared
- Trained model artifact

This establishes a reproducible baseline for comparison with the Random Forest model.

In [7]:
# ------------------------------------------------------------
# Task 4.6 - Linear Regression MLflow run
# ------------------------------------------------------------

linear_model = LinearRegression()

linear_model.fit(
    X_train_scaled,
    y_train
)

linear_predictions = linear_model.predict(
    X_test_scaled
)

linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)


with mlflow.start_run(
    run_name="linear_regression_baseline"
) as linear_run:

    # Parameters
    mlflow.log_param(
        "algorithm",
        "LinearRegression"
    )

    mlflow.log_param(
        "test_size",
        0.20
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_param(
        "number_of_features",
        len(feature_columns)
    )

    mlflow.log_param(
        "feature_scaling",
        True
    )

    # Metrics
    mlflow.log_metric(
        "MAE",
        linear_mae
    )

    mlflow.log_metric(
        "R_squared",
        linear_r2
    )

    # Model artifact
    mlflow.sklearn.log_model(
        linear_model,
        name="model"
    )

    linear_run_id = linear_run.info.run_id


logger.info(
    "Linear Regression logged to MLflow: "
    "run_id=%s, MAE=%.4f, R2=%.4f.",
    linear_run_id,
    linear_mae,
    linear_r2
)


print("Linear Regression MLflow Run")
print("-" * 50)
print("Run ID:", linear_run_id)
print(f"MAE: {linear_mae:.4f}")
print(f"R-squared: {linear_r2:.4f}")

2026/10/07 22:18:37 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/07 22:18:37 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Linear Regression MLflow Run
--------------------------------------------------
Run ID: f3cd1322587143d896b9acd97122d8fb
MAE: 813.7666
R-squared: 0.7259


## 4.7 Track the Random Forest Regressor

A Random Forest Regressor is recorded as the second MLflow run.

The model uses the same train/test split and prediction target as the Linear Regression baseline, allowing a direct comparison of model performance.

In addition to the common experiment parameters and metrics, the Random Forest-specific parameters are recorded.

In [9]:
# ------------------------------------------------------------
# Task 4.7 - Random Forest MLflow run
# ------------------------------------------------------------

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_predictions = rf_model.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)


with mlflow.start_run(
    run_name="random_forest_regressor"
) as rf_run:

    # Parameters
    mlflow.log_param(
        "algorithm",
        "RandomForestRegressor"
    )

    mlflow.log_param(
        "n_estimators",
        100
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_param(
        "test_size",
        0.20
    )

    mlflow.log_param(
        "number_of_features",
        len(feature_columns)
    )

    mlflow.log_param(
        "feature_scaling",
        False
    )

    # Metrics
    mlflow.log_metric(
        "MAE",
        rf_mae
    )

    mlflow.log_metric(
        "R_squared",
        rf_r2
    )

    # Model artifact
    # cloudpickle is used explicitly for compatibility
    # with the Random Forest estimator.
    mlflow.sklearn.log_model(
        rf_model,
        name="model",
        serialization_format="cloudpickle"
    )

    rf_run_id = rf_run.info.run_id


logger.info(
    "Random Forest logged to MLflow: "
    "run_id=%s, MAE=%.4f, R2=%.4f.",
    rf_run_id,
    rf_mae,
    rf_r2
)


print("Random Forest MLflow Run")
print("-" * 50)
print("Run ID:", rf_run_id)
print(f"MAE: {rf_mae:.4f}")
print(f"R-squared: {rf_r2:.4f}")

2026/10/07 22:26:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026-10-07 22:26:56,590 - INFO - __main__ - Random Forest logged to MLflow: run_id=9b06e1bd74c74c6fb7197c2c7c8ec520, MAE=271.3004, R2=0.9442.


Random Forest MLflow Run
--------------------------------------------------
Run ID: 9b06e1bd74c74c6fb7197c2c7c8ec520
MAE: 271.3004
R-squared: 0.9442


## 4.8 Compare MLflow Experiments

The tracked MLflow runs are retrieved programmatically to confirm that the experiment contains the required model parameters and evaluation metrics.

Lower MAE and higher R-squared indicate stronger regression performance.

In [11]:
# ------------------------------------------------------------
# Task 4.8 - Retrieve completed MLflow experiment results
# ------------------------------------------------------------

experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

runs = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ]
)


# ------------------------------------------------------------
# Retain only successfully completed runs
# ------------------------------------------------------------

completed_runs = runs.loc[
    runs["status"] == "FINISHED"
].copy()


comparison_columns = [
    "tags.mlflow.runName",
    "params.algorithm",
    "metrics.MAE",
    "metrics.R_squared",
    "run_id",
    "status"
]


mlflow_comparison = (
    completed_runs[
        comparison_columns
    ]
    .rename(
        columns={
            "tags.mlflow.runName": "Run_Name",
            "params.algorithm": "Algorithm",
            "metrics.MAE": "MAE",
            "metrics.R_squared": "R_squared",
            "run_id": "Run_ID",
            "status": "Status"
        }
    )
    .sort_values(
        "MAE"
    )
    .reset_index(drop=True)
)


logger.info(
    "Retrieved %d completed MLflow runs for comparison.",
    len(mlflow_comparison)
)


mlflow_comparison

2026-10-07 22:31:09,555 - INFO - __main__ - Retrieved 2 completed MLflow runs for comparison.


,Run_Name,Algorithm,MAE,R_squared,Run_ID,Status
0,random_forest_regressor,RandomForestRegressor,271.300362,0.944176,9b06e1bd74c74c6fb7197c2c7c8ec520,FINISHED
1,linear_regression_baseline,LinearRegression,813.766579,0.725922,f3cd1322587143d896b9acd97122d8fb,FINISHED


## 4.9 Why MLflow Was Selected, Implementation, Value and Limitations

### Why MLflow Was Selected

MLflow was selected as the advanced AI technique because the capstone contains multiple machine learning models that need to be compared, reproduced and managed consistently.

Without experiment tracking, model parameters, evaluation metrics and model files can become separated across notebooks and scripts. MLflow provides a structured record of each experiment and therefore connects naturally with the MLOps requirements developed later in Task 6.

### How MLflow Was Implemented

A local MLflow experiment named `capstone_traffic_volume_prediction` was created using a SQLite tracking database.

Two regression models were tracked:

1. **Linear Regression** – baseline model
2. **Random Forest Regressor** – non-linear ensemble model

For each completed run, MLflow recorded:

- Algorithm name
- Train/test configuration
- Number of predictors
- Model-specific parameters
- Mean Absolute Error (MAE)
- R-squared (R²)
- Trained model artifact
- Environment and dependency metadata

The completed runs produced the following results:

| Model | MAE | R-squared |
|---|---:|---:|
| Linear Regression | 813.7666 | 0.7259 |
| Random Forest Regressor | 271.3004 | 0.9442 |

The Random Forest Regressor achieved substantially lower prediction error and higher explanatory performance than the Linear Regression baseline.

The Random Forest MAE of **271.3004** differs slightly from the Task 1 value of **271.21** because this run was logged independently, although it used the same split and random state. The difference is small and does not change the model comparison: Random Forest remains the stronger regression model.

An initial Random Forest model-artifact logging attempt failed because the default `skops` serialisation process identified an internal scikit-learn tree type as untrusted. The successful run therefore used `cloudpickle` serialisation. Only successfully completed (`FINISHED`) runs were included in the formal model comparison.

### Value Added

MLflow adds value to the traffic intelligence solution by creating a persistent and reproducible record of model development.

Instead of relying only on notebook outputs, each experiment has a run ID linking its configuration, metrics and trained model artifact. This makes it easier to:

- Compare alternative models
- Reproduce experiments
- Identify the best-performing model
- Preserve trained model artifacts
- Support model versioning
- Provide evidence of model development
- Prepare models for later deployment and monitoring workflows

The experiment therefore provides a foundation for the model-versioning and experiment-tracking components of the MLOps workflow in Task 6.

### Limitations

The current implementation uses a local SQLite MLflow tracking database. This is appropriate for a single-user capstone project but would not provide the scalability, access control, backup strategy or collaborative capabilities required for a production environment.

The MLflow runs produced warnings because Git metadata could not be collected automatically from the current Jupyter environment. The model parameters, metrics and artifacts were still logged successfully, but Git commit information was not attached automatically to these runs.

The Random Forest model artifact was stored using `cloudpickle` because the default `skops` serialisation rejected an internal tree type. Pickle-based formats should only be loaded from trusted sources because Python object deserialisation can execute code.

Finally, MLflow improves experiment management and reproducibility but does not by itself guarantee that a model is unbiased, generalises to future data or remains accurate after deployment. These issues require separate validation, governance and monitoring processes.



In [13]:
# ------------------------------------------------------------
# Task 4.10 - Final validation
# ------------------------------------------------------------

experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

all_runs = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ]
)

finished_runs = all_runs.loc[
    all_runs["status"] == "FINISHED"
].copy()


required_metrics = {
    "metrics.MAE",
    "metrics.R_squared"
}

required_params = {
    "params.algorithm",
    "params.test_size",
    "params.number_of_features"
}


experiment_exists = (
    experiment is not None
)

two_completed_runs = (
    len(finished_runs) >= 2
)

metrics_complete = (
    required_metrics
    .issubset(finished_runs.columns)
    and finished_runs[
        list(required_metrics)
    ].notna().all().all()
)

parameters_complete = (
    required_params
    .issubset(finished_runs.columns)
    and finished_runs[
        list(required_params)
    ].notna().all().all()
)

linear_run_complete = (
    finished_runs[
        "params.algorithm"
    ]
    .eq("LinearRegression")
    .any()
)

rf_run_complete = (
    finished_runs[
        "params.algorithm"
    ]
    .eq("RandomForestRegressor")
    .any()
)

tracking_database_exists = (
    MLFLOW_DB.exists()
)


task4_validation = {
    "MLflow experiment exists":
        experiment_exists,

    "At least two completed runs tracked":
        two_completed_runs,

    "Required metrics logged":
        metrics_complete,

    "Required parameters logged":
        parameters_complete,

    "Linear Regression run completed":
        linear_run_complete,

    "Random Forest run completed":
        rf_run_complete,

    "MLflow tracking database exists":
        tracking_database_exists
}


logger.info(
    "Task 4 final validation completed."
)


print("TASK 4 FINAL VALIDATION")
print("-" * 60)

for check, passed in task4_validation.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check}: {status}")

all_checks_passed = all(
    task4_validation.values()
)

print("-" * 60)

if all_checks_passed:
    print(
        "TASK 4 STATUS: PASS - "
        "All required checks completed."
    )
else:
    print(
        "TASK 4 STATUS: FAIL - "
        "Review the failed checks."
    )

2026-10-07 23:05:19,319 - INFO - __main__ - Task 4 final validation completed.


TASK 4 FINAL VALIDATION
------------------------------------------------------------
MLflow experiment exists: PASS
At least two completed runs tracked: PASS
Required metrics logged: PASS
Required parameters logged: PASS
Linear Regression run completed: PASS
Random Forest run completed: PASS
MLflow tracking database exists: PASS
------------------------------------------------------------
TASK 4 STATUS: PASS - All required checks completed.


## 4.11 Task 4 Summary

Task 4 applied MLflow as the advanced AI technique for experiment tracking and model management.

A local MLflow experiment named `capstone_traffic_volume_prediction` was created using a SQLite tracking database. Linear Regression and Random Forest Regression were tracked using the same traffic-volume prediction problem.

The Linear Regression baseline achieved an MAE of **813.7666** and an R-squared value of **0.7259**. The Random Forest Regressor achieved an MAE of **271.3004** and an R-squared value of **0.9442**, making it the stronger of the two tracked models.

The Random Forest MAE of **271.3004** differs slightly from the Task 1 value of **271.21** because the MLflow experiment was logged independently, although it used the same split and random state.

MLflow successfully recorded model parameters, evaluation metrics, run identifiers and trained model artifacts. The Random Forest artifact also includes environment and dependency metadata to support reproducibility.

An initial Random Forest artifact-logging attempt failed because the default `skops` serialisation process rejected an internal scikit-learn tree type. A subsequent completed run used `cloudpickle` serialisation successfully. Formal model comparison therefore includes only MLflow runs with `FINISHED` status.

The MLflow implementation provides a reproducible experiment history and establishes a foundation for the model-versioning, experiment-tracking, deployment and monitoring workflow developed in Task 6.

The current implementation uses a local SQLite tracking store and is intended as an MLOps simulation rather than a production-scale tracking infrastructure.